## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

MASKSDM_DIR = '/content/MaskSDM-MEE'
DATA_DIR    = '/content/drive/MyDrive/CISO/data'
CKPT_DIR    = '/content/drive/MyDrive/MaskSDM/model_checkpoints_1339'
RESULTS_DIR = '/content/drive/MyDrive/MaskSDM/results_plants_1000_epochs_1339'

for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(MASKSDM_DIR):
    subprocess.run(
        ['git', 'clone', 'https://github.com/zbirobin/MaskSDM-MEE', MASKSDM_DIR],
        check=True
    )
    print('Repo cloned.')
else:
    print('Repo already cloned, skipping.')

os.chdir(MASKSDM_DIR)
print('Working directory:', os.getcwd())
print('\nSetup complete.')

In [ ]:
!pip install verde schedulefree elapid torcheval -q

In [ ]:
import sys
sys.path.insert(0, MASKSDM_DIR)

import numpy as np
import pandas as pd
import json
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

from data_helpers import get_torch_dataset
from modules import get_model
from training_helpers import seed_everything, train

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Imports OK')

## 1. Load Data

In [ ]:
DATA_CSV    = f'{DATA_DIR}/splotopen_global.csv'
SPLITS_JSON = f'{DATA_DIR}/splotopen_global_splits.json'

for f in [DATA_CSV, SPLITS_JSON]:
    print('OK  ' if os.path.exists(f) else 'MISSING  ', f)

plants = pd.read_csv(DATA_CSV)
plants = plants.reset_index(drop=True)

with open(SPLITS_JSON) as f:
    plants_splits = json.load(f)

print(f'\nLoaded {len(plants):,} rows x {len(plants.columns):,} columns')
print(f'Split keys: {list(plants_splits.keys())}')
plants.head(2)

In [ ]:
env_cols     = [c for c in plants.columns if c.startswith('env_')]
species_cols = [c for c in plants.columns
                if c not in env_cols + ['time', 'latitude', 'longitude']]

worldclim_cols = [c for c in env_cols if 'bio' in c.lower()]
soilgrid_cols  = [c for c in env_cols if 'bio' not in c.lower()]

print(f'WorldClim cols ({len(worldclim_cols)}): {worldclim_cols}')
print(f'SoilGrid cols  ({len(soilgrid_cols)}):  {soilgrid_cols}')
print(f'Species cols   ({len(species_cols)}):   first 5 = {species_cols[:5]}')

## 2. Build Split Indices and Data Arrays

In [ ]:
train_split = np.array(plants_splits['train'])
val_split   = np.array(plants_splits['val'])
test_split  = np.array(plants_splits['test'])

assert train_split.max() < len(plants), 'Train index out of bounds'
assert val_split.max()   < len(plants), 'Val index out of bounds'
assert test_split.max()  < len(plants), 'Test index out of bounds'
print(f'train={len(train_split):,}  val={len(val_split):,}  test={len(test_split):,}')
print('All indices in bounds.')

In [ ]:
# Filter species: >= 100 occurrences
targets_full   = plants[species_cols].to_numpy().astype(np.float32)
species_counts = targets_full.sum(axis=0)
keep           = species_counts >= 100
targets        = targets_full[:, keep]
species_cols_filtered = [s for s, k in zip(species_cols, keep) if k]

print(f'Species before filtering: {len(species_cols):,}')
print(f'Species after  filtering: {len(species_cols_filtered):,}')

tabular_x = plants[env_cols].to_numpy().astype(np.float32)

# SatCLIP embeddings are unavailable; zeros are masked during training.
N_SATCLIP = 256
satclip_embeddings = np.zeros((len(plants), N_SATCLIP), dtype=np.float32)

data = {
    'tabular_x':          tabular_x,
    'y':                  targets,
    'satclip_embeddings': satclip_embeddings,
}

data['x_train'] = tabular_x[train_split]
data['x_val']   = tabular_x[val_split]
data['x_test']  = tabular_x[test_split]
data['y_train'] = targets[train_split]
data['y_val']   = targets[val_split]
data['y_test']  = targets[test_split]
data['satclip_embeddings_train'] = satclip_embeddings[train_split]
data['satclip_embeddings_val']   = satclip_embeddings[val_split]
data['satclip_embeddings_test']  = satclip_embeddings[test_split]

# Normalize using training statistics.
train_mean = np.nanmean(data['x_train'], axis=0)
train_std  = np.nanstd(data['x_train'],  axis=0)
data['x_train'] = (data['x_train'] - train_mean) / (train_std + 1e-4)
data['x_val']   = (data['x_val']   - train_mean) / (train_std + 1e-4)
data['x_test']  = (data['x_test']  - train_mean) / (train_std + 1e-4)

print(f'x_train: {data["x_train"].shape}')
print(f'x_val:   {data["x_val"].shape}')
print(f'x_test:  {data["x_test"].shape}')

## 3. Verify Integrity

In [ ]:
n_features = data['x_train'].shape[1]
n_species  = data['y_train'].shape[1]

# Species that appear in all three splits — used for evaluation
indices_evaluated = np.intersect1d(
    np.intersect1d(
        data['y_train'].sum(axis=0).nonzero()[0],
        data['y_val'].sum(axis=0).nonzero()[0]
    ),
    data['y_test'].sum(axis=0).nonzero()[0]
).tolist()

print(f'n_features:        {n_features}')
print(f'n_species:         {n_species}')
print(f'species_evaluated: {len(indices_evaluated)}')
print(f'train rows:        {len(data["x_train"]):,}')
print(f'val rows:          {len(data["x_val"]):,}')
print(f'test rows:         {len(data["x_test"]):,}')
print('\nAll checks passed.')

## 4. Build Config and Train

In [ ]:
random_seed = 1339
seed_everything(random_seed)
torch.set_default_device(device)

# Edit these
MAX_EPOCHS = 1000
BATCH_SIZE = 256
SAVE_DIR   = f'{CKPT_DIR}/masksdm_splot'

config = {
    'device':                    device,
    'seed':                      random_seed,
    'dataset':                   'splot',
    'n_features':                n_features,
    'n_species':                 n_species,
    'n_samples_train':           len(data['y_train']),
    'n_samples_val':             len(data['y_val']),
    'n_samples_test':            len(data['y_test']),
    'indices_evaluated_species': indices_evaluated,
    'n_evaluated_species':       len(indices_evaluated),
    # satclip=False: zeros are always fully masked so they carry no signal
    'satclip':                   False,
    'model':                     'FTTransformer',
    'd_hidden':                  192,
    'n_heads':                   8,
    'n_blocks':                  7,
    'n_layers':                  7,
    'dropout':                   0.1,
    'd_out':                     n_species,
    'epochs':                    MAX_EPOCHS,
    'batch_size':                BATCH_SIZE,
    'batch_size_eval':           4096,
    'loss':                      'weighted',
    'species_weights':           torch.tensor(
                                     len(data['y_train']) / (data['y_train'].sum(axis=0) + 1e-5),
                                     dtype=torch.float32
                                 ).to(device),
    'optimizer':                 'AdamW',
    'scheduler_free':            True,
    'lr':                        0.001,
    'weight_decay':              0.01,
    'warmup_steps':              1000,
    'masking':                   True,
    'extra_masking':             True,
    'save_dir':                  SAVE_DIR,
    'use_wandb':                 False,
    'wandb_init':                {},
}

print(f'n_features:  {n_features}')
print(f'n_species:   {n_species}')
print(f'max_epochs:  {MAX_EPOCHS}')
print(f'save_dir:    {SAVE_DIR}')
print(f'device:      {device}')

In [ ]:
train(config, data)

In [ ]:
BEST_EPOCH = 46
SAVE_DIR   = f'{CKPT_DIR}/masksdm_splot'
CHECKPOINT_PATH = f"{SAVE_DIR}/epoch_{BEST_EPOCH}.pt"

print(f"Checkpoint: {CHECKPOINT_PATH}")

## 5. Inference with 100% Masking (p=1.0)

All tabular predictors are masked — the model receives only the learned mask token for every input.
This matches STEM-LM p=1.0 (fully unconditioned).
MaskSDM is trained with masked data modelling so this is the setting it is designed to handle gracefully.

In [ ]:
model = get_model(config).to(device)
state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print('Model loaded.')

In [ ]:
test_loader = DataLoader(
    get_torch_dataset(config, data['x_test'], data['y_test'], data['satclip_embeddings_test']),
    batch_size=config['batch_size_eval'],
    shuffle=False,
)

all_probs, all_targets = [], []

with torch.no_grad():
    for batch in test_loader:
        x_batch, y_batch, satclip_emb = batch
        x_batch     = x_batch.to(device)
        satclip_emb = satclip_emb.to(device)

        # Match upstream MaskSDM evaluate(): keep all valid features (mask=1=kept).
        # SatCLIP stays masked because we did not train with SatCLIP embeddings.
        x_mask       = ~torch.isnan(x_batch).to(device)
        satclip_mask = torch.zeros(len(satclip_emb), dtype=torch.bool, device=device)

        logits = model(x_batch, satclip_emb, x_mask, satclip_mask)
        probs  = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

probs   = np.concatenate(all_probs,   axis=0)                    # (N_test, S)
targets = np.concatenate(all_targets, axis=0).astype(np.int64)   # (N_test, S)

print(f'probs:   {probs.shape}')
print(f'targets: {targets.shape}')

## 6. STEM-LM Metrics

In [ ]:

def _safe_auc_roc(y, p):
    if y.size == 0 or len(set(y.tolist())) < 2 or np.isnan(p).any(): return float('nan')
    try: return float(roc_auc_score(y, p))
    except Exception: return float('nan')

def _safe_auc_pr(y, p):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    try: return float(average_precision_score(y, p))
    except Exception: return float('nan')

def _safe_brier(y, p):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    return float(np.mean((p - y.astype(np.float64))**2))

def _safe_ece(y, p, n_bins=15):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    edges = np.linspace(0, 1, n_bins+1)
    idx   = np.clip(np.digitize(p, edges) - 1, 0, n_bins-1)
    err = 0.0; n = p.size
    for b in range(n_bins):
        m = idx == b
        if not m.any(): continue
        err += (m.sum()/n) * abs(y[m].mean() - p[m].mean())
    return float(err)

def _safe_cbi(y, p, n_windows=101, width=0.1, min_per_window=10):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    pres = p[y==1]; bg = p[y==0]
    if pres.size == 0 or bg.size == 0: return float('nan')
    centers = np.linspace(0, 1, n_windows); half = width/2
    pe = np.full(n_windows, np.nan)
    for i, c in enumerate(centers):
        lo, hi = c-half, c+half
        n_bg = int(((bg>=lo)&(bg<=hi)).sum())
        if n_bg < min_per_window: continue
        e = n_bg/bg.size
        if e == 0: continue
        pe[i] = ((pres>=lo)&(pres<=hi)).sum()/pres.size / e
    ok = np.isfinite(pe)
    if ok.sum() < 3 or np.unique(pe[ok]).size < 2: return float('nan')
    try:
        rho = spearmanr(centers[ok], pe[ok]).statistic
        return float(rho) if np.isfinite(rho) else float('nan')
    except Exception: return float('nan')

print('Metric functions defined.')

In [ ]:
# At p=1.0 every feature is masked for every row, so we use all test rows.
# Per-split species filter: skip species with no presences (or no absences) in test
# — matches how the R baselines (Logistic / GAM / Maxnet) report n_species.
auc_roc_vals, auc_pr_vals, cbi_vals, brier_vals, ece_vals = [], [], [], [], []

for s in range(targets.shape[1]):
    y = targets[:, s].astype(np.int64)
    p = probs[:, s].astype(np.float64)
    if y.sum() == 0 or y.sum() == len(y):
        continue
    auc_roc_vals.append(_safe_auc_roc(y, p))
    auc_pr_vals.append(_safe_auc_pr(y, p))
    cbi_vals.append(_safe_cbi(y, p))
    brier_vals.append(_safe_brier(y, p))
    ece_vals.append(_safe_ece(y, p))

def _nanmean(v):   return float(np.nanmean([x for x in v if np.isfinite(x)])) if v else float('nan')
def _nanq(v, q):   vals=[x for x in v if np.isfinite(x)]; return float(np.quantile(vals,q)) if vals else float('nan')

summary = {
    'model':          'MaskSDM',
    'masking_p':      1.0,
    'eval_known_ratio': 0.0,
    'n_species':      len(auc_roc_vals),
    'mean_auc_roc':   _nanmean(auc_roc_vals),
    'auc_roc_q25':    _nanq(auc_roc_vals, 0.25),
    'auc_roc_q50':    _nanq(auc_roc_vals, 0.50),
    'auc_roc_q75':    _nanq(auc_roc_vals, 0.75),
    'mean_auc_pr':    _nanmean(auc_pr_vals),
    'mean_cbi':       _nanmean(cbi_vals),
    'mean_brier':     _nanmean(brier_vals),
    'mean_ece':       _nanmean(ece_vals),
}

cols = ['mean_auc_roc','auc_roc_q25','auc_roc_q50','auc_roc_q75',
        'mean_auc_pr','mean_cbi','mean_brier','mean_ece','n_species']

print('\n-- MaskSDM Benchmark (p=1.0, fully unconditioned) --')
for k in cols:
    v = summary[k]
    print(f'  {k:<20} {round(v, 4) if isinstance(v, float) else v}')

In [ ]:
import json as _json

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

out_csv  = f'{RESULTS_DIR}/masksdm_benchmark_summary.csv'
out_json = f'{RESULTS_DIR}/masksdm_benchmark_summary.json'

pd.DataFrame([summary]).set_index('masking_p').to_csv(out_csv)
with open(out_json, 'w') as f:
    _json.dump(summary, f, indent=2)

print(f'CSV  saved to {out_csv}')
print(f'JSON saved to {out_json}')